# 05 — LoRA SFT of Gemma with Completion-Only Loss · Google Colab

**Phase 2 / Month 2** · MSc Thesis — ECLIPSE project  
Supervisor: Dr. Panagiotis Kasnesis | Student: Antonios Bastoulis

---

**Setup follows the official Unsloth Gemma-4 tutorial.** The install block (§ 1)
pins the exact versions Gemma 4 expects — `transformers==5.5.0` (unsloth-zoo
hard-caps it at `<=5.5.0`; an unpinned `--upgrade` pulls 5.8.x and the Gemma-4
patch breaks), a `torch`-matched `xformers`, `datasets==4.3.0`, and `timm`. This
keeps training rendering the *same* chat template the model was tuned against.

The loss is **two-level masked**, both masks built in one tokenisation `.map`
(§ 6). (1) *Completion-only*: the ~480-token system prompt is masked so
cross-entropy is computed only on the assistant turn — without it the prompt
would dilute the ~10-20-token action target ~25×. (2) *No-op cells*: within the
assistant turn, the `<action>` line of any building whose SAC move is a physical
no-op (discharge from an empty battery / charge into a full one — clipped to
nothing by CityLearn) is also masked. Those cells carry an artifact `IDLE`
label, not a teacher decision; supervising them collapsed the first run to
constant IDLE. This replaces Unsloth's `train_on_responses_only`, which does
mask (1) only.

Pipeline:
1. Pull the SAC distillation JSONL produced by 04.
2. Load Gemma-4 E4B via **Unsloth** `FastModel` — 4-bit + LoRA.
3. Filter no-op rows + flag no-op cells (`filter_uninformative_rows`, `attach_supervision_masks`).
4. SFT with TRL's `SFTTrainer`; loss labels built by a custom tokenisation map.
5. Evaluate vs. no-control + RBC on a 1-week mid-dataset slice.


## § 0 — Runtime & Config
> **Edit this cell only.** Nothing else needs changing.

In [ ]:
import os, sys, subprocess, time, warnings, json, random
import numpy as np

# ── Repo + dataset selection ──────────────────────────────────────────────
GITHUB_REPO    = "https://github.com/antonisbast/eclipse-thesis.git"  # adjust if needed
REPO_DIR       = "/content/eclipse-thesis"
DATASET_GLOB   = "notebooks/artifacts/sft_datasets/sac_*.jsonl"  # picks newest match

# ── Model selection ───────────────────────────────────────────────────────
# Unsloth-optimized Gemma 4 E4B
MODEL_ID:       str  = "unsloth/gemma-4-E4B-it"
LOAD_IN_4BIT:   bool = True
MAX_NEW_TOKENS: int  = 80      # action block is ~30-50 tokens; 80 leaves headroom

# ── LoRA / SFT hyperparameters ────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.0    # Unsloth recommends 0 for fastest path
EPOCHS         = 1      # was 3 — loss flattened at step ~200/1866 in v3 run
BATCH_SIZE     = 2
GRAD_ACCUM     = 8      # effective batch 16
LEARNING_RATE  = 5e-5
# Keep model max_seq_length aligned with SFTConfig.max_length so we don't
# allocate activation buffers for tokens we never use. Sample length is
# ~490 tokens; 1024 leaves plenty of headroom to prevent truncation.
MAX_SEQ_LEN    = 1024

# ── Evaluation slice ──────────────────────────────────────────────────────
EVAL_START     = 1440           # mid-summer slice, batteries primed by then
EVAL_LEN       = 168            # 1-week eval; raise to 8760 for full year

# ── CityLearn version ────────────────────────────────────────────────────
# Pin 2.6.0b2 so src.eval.evaluate_v2() works (matches nb 04 / 06).
CITYLEARN_VERSION = "2.6.0b2"

# ── Misc ──────────────────────────────────────────────────────────────────
HF_TOKEN: str = ""
SEED          = 42

try:
    import torch
    if torch.cuda.is_available():
        _gpu  = torch.cuda.get_device_name(0)
        _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ GPU: {_gpu}  ({_vram:.1f} GB VRAM)")
    else:
        print("⚠  No GPU — set Runtime → Change runtime type → T4 GPU")
except ImportError:
    print("torch not yet installed — will be in § 1")


✓ GPU: Tesla T4  (15.6 GB VRAM)


## § 1 — Install Dependencies

In [ ]:
# CityLearn 2.6 is a pre-release on PyPI — pin the version explicitly so we
# get the same evaluate_v2() API as the rest of the pipeline (nb 04 / 06).
# --no-deps because CityLearn pulls heavy/legacy deps we don't need at eval
# time (e.g. some EnergyPlus extras). Runtime deps installed explicitly.
!pip install -q numpy gymnasium doe-xstock nrel-pysam
!pip install -q --pre "CityLearn=={CITYLEARN_VERSION}" --no-deps

import citylearn
assert citylearn.__version__.startswith("2.6"), (
    f"Expected CityLearn 2.6.x, got {citylearn.__version__}. "
    f"src.eval depends on evaluate_v2() which only exists in 2.6+."
)
print(f"✓ CityLearn {citylearn.__version__}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.6/423.6 kB 16.6 MB/s eta 0:00:00
✓ CityLearn 2.6.0b2


In [ ]:
# Unsloth + Gemma-4 SFT stack — versions pinned to match the OFFICIAL Unsloth
# Gemma-4 tutorial. This matters: Gemma 4 + Unsloth's patch require
# transformers==5.5.0 (unsloth-zoo hard-caps it at <=5.5.0). An unpinned
# `pip install --upgrade transformers` pulls 5.8.x and the Gemma-4 patch breaks.
import os, re, torch

# xformers wheel must match the installed torch — same table the tutorial uses.
_v = re.match(r"[\d]+\.[\d]+", str(torch.__version__)).group(0)
_xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1",
                            "2.8": "0.0.32.post2"}.get(_v, "0.0.34")

!pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {_xformers} peft trl triton unsloth
!pip install -q --no-deps --upgrade "torchao>=0.16.0"
# transformers 5.5.0 + matched tokenizers — installed LAST, --no-deps, so no
# upstream package silently bumps them past Unsloth's supported ceiling.
!pip install -q --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install -q torchcodec
# timm is needed by Gemma 4 — E4B carries vision/audio towers even when we
# only fine-tune the text layers.
!pip install -q --no-deps --upgrade timm

torch._dynamo.config.recompile_limit = 64

import unsloth          # triggers Unsloth's model patching
import trl
# Completion-only loss is applied via Unsloth's train_on_responses_only (§ 6),
# NOT DataCollatorForCompletionOnlyLM — so TRL is left unpinned, like the tutorial.
from unsloth.chat_templates import train_on_responses_only  # noqa: F401
import transformers
print(f"✓ Unsloth + SFT stack installed")
print(f"  transformers=={transformers.__version__}  (expect 5.5.0)  |  trl=={trl.__version__}")
assert transformers.__version__.startswith("5.5"), (
    f"transformers is {transformers.__version__}, expected 5.5.x — "
    f"unsloth-zoo caps it at <=5.5.0; re-run this cell."
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 47.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.7/842.7 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 105.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## § 1.5 — Optional Performance Dep (best-effort)

`cut_cross_entropy` is Unsloth's memory-efficient cross-entropy — with it the
effective batch can grow without OOM. `unsloth-zoo` lists it as an optional
extra (not pulled in by the `--no-deps` install in § 1), so we install it here.
The install is wrapped so a failure does not brick the notebook — Unsloth falls
back to standard CE.

Attention kernels (`xformers`) and `hf_transfer` (parallel HF downloads) were
already installed with matched versions in § 1; this cell just activates
`hf_transfer` and reports what made it in.


In [ ]:
# Optional perf dep — best-effort. A failure here does NOT break the notebook.
import subprocess, sys, importlib, os

# hf_transfer was installed in § 1 — activate the parallel-download path.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# cut_cross_entropy: Unsloth's memory-efficient CE (optional unsloth-zoo extra).
try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "cut_cross_entropy"],
        check=True, timeout=300,
    )
    print("✓ cut_cross_entropy installed")
except Exception as e:
    print(f"✗ cut_cross_entropy ({type(e).__name__}) — Unsloth uses standard CE, fine")

# Report what is actually importable.
print("\nPerf backend availability:")
for mod in ["xformers", "cut_cross_entropy", "hf_transfer"]:
    try:
        importlib.import_module(mod)
        print(f"  ✓ {mod}")
    except Exception as e:
        print(f"  ✗ {mod}  ({type(e).__name__})")


✓ cut_cross_entropy installed

Perf backend availability:
  ✓ xformers
  ✓ cut_cross_entropy
  ✓ hf_transfer


## § 2 — Clone Repository (pulls `src/sft.py` and the JSONL dataset)

In [ ]:
if not os.path.exists(REPO_DIR):
    res = subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR],
                         capture_output=True, text=True)
    print(res.stdout or res.stderr)
else:
    res = subprocess.run(["git", "-C", REPO_DIR, "pull"],
                         capture_output=True, text=True)
    print("Repo present —", res.stdout.strip() or "up to date")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

Cloning into '/content/eclipse-thesis'...



## § 3 — Load Distillation Dataset

In [ ]:
import glob
from pathlib import Path

from src.sft import filter_uninformative_rows, attach_supervision_masks

candidates = sorted(glob.glob(os.path.join(REPO_DIR, DATASET_GLOB)))
assert candidates, f"No JSONL matching {DATASET_GLOB} in repo. Run notebook 04 + push first."
JSONL_PATH = candidates[-1]
print(f"Using dataset: {JSONL_PATH}")

raw_rows = [json.loads(l) for l in Path(JSONL_PATH).read_text().strip().split("\n")]
print(f"Raw examples : {len(raw_rows)}")

# Drop rows where EVERY building's action is a physical no-op
# (discharge from SoC≈0, charge into SoC≈100, or |a|<0.05).
# Removes ~51% of rows; IDLE share of action tokens drops ~73% -> ~45%.
raw_rows = filter_uninformative_rows(raw_rows)
print(f"After filter : {len(raw_rows)}")

# Per-cell supervision mask: flag each building's action as supervised or
# masked. A cell is MASKED when SAC's move is a physical no-op (discharge
# from an empty battery / charge into a full one) — its IDLE label is an
# artifact of CityLearn's action clipping, not a teacher decision. Masking
# those cells (vs supervising them) drops the IDLE share of *supervised*
# tokens from ~45% to ~10%; the previous run supervised them and collapsed
# to constant IDLE. Rows where EVERY building is a no-op are dropped here.
raw_rows = attach_supervision_masks(raw_rows)
_cells = sum(len(r["supervise"]) for r in raw_rows)
_sup   = sum(sum(r["supervise"]) for r in raw_rows)
print(f"After mask   : {len(raw_rows)} rows | "
      f"{_sup}/{_cells} cells supervised ({_sup / _cells * 100:.0f}%), "
      f"{_cells - _sup} no-op cells masked from loss")

print("\nFirst row preview:")
print("  prompt   :", raw_rows[0]["prompt"][:200].replace("\n", " | "))
print("  response :", raw_rows[0]["response"][:200].replace("\n", " | "))
print("  supervise:", raw_rows[0]["supervise"])


Using dataset: /content/eclipse-thesis/notebooks/artifacts/sft_datasets/sac_merlin_distill_20260517_121538.jsonl
Raw examples : 17518
After filter : 8500
After mask   : 8367 rows | 15506/25101 cells supervised (62%), 9595 no-op cells masked from loss

First row preview:
  prompt   : STATE: | Aug, Mon 08:00  |  price=LOW  |  carbon=HIGH | Buildings: |   B0: SoC=  0.0%  load=0.63 kWh  last_net=-0.49 kWh  solar=MID |   B1: SoC=  0.0%  load=1.65 kWh  last_net=+1.12 kWh  solar=MID |   B2: SoC= 
  response : <action building=0>CHARGE_20</action> | <action building=1>IDLE</action> | <action building=2>CHARGE_20</action>
  supervise: [True, False, True]


## § 4 — Load Gemma + Attach LoRA (Unsloth `FastModel`)

`FastModel.from_pretrained` returns the model **and** tokenizer with Unsloth's
kernels patched in. `FastModel.get_peft_model` attaches LoRA adapters in
Unsloth-optimized form (no `prepare_model_for_kbit_training` needed).

In [ ]:
from unsloth import FastModel
import torch
import time

# Clear any lingering memory from previous failed runs
torch.cuda.empty_cache()

_t0 = time.time()
model, tokenizer = FastModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,  # Let Unsloth auto-detect optimal precision for Gemma 4
    load_in_4bit    = LOAD_IN_4BIT,
    full_finetuning = False,
    token           = HF_TOKEN or None,
)
print(f"Base model loaded in {time.time()-_t0:.1f}s")

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # text-only distillation
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    random_state   = SEED,
)
model.print_trainable_parameters()

==((====))==  Unsloth 2026.5.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Base model loaded in 61.4s
trainable params: 36,700,160 || all params: 8,032,856,608 || trainable%: 0.4569


## § 5 — Build HF Dataset with Gemma Chat Template

Gemma rejects the `system` role, so the SFT prompt is merged into the user
turn (same workaround as notebook 03's `LocalHFProvider`).

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

from src.sft import make_sft_prompt

# Use Unsloth's curated Gemma-4 chat template so training AND inference render
# the exact template the model was tuned against (parity with the official
# Unsloth Gemma-4 tutorial). Reassigns the global `tokenizer`.
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

N_BUILDINGS   = 3
SYSTEM_PROMPT = make_sft_prompt(N_BUILDINGS)

# Pin padding side — causal LM training requires right-padding so that
# the loss is computed on real tokens, not on left-pad noise. Default
# differs across tokenizer versions, so set it explicitly.
text_tok = getattr(tokenizer, "tokenizer", tokenizer)
text_tok.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_is_gemma = "gemma" in MODEL_ID.lower()

def to_chat(row):
    if _is_gemma:
        msgs = [
            {"role": "user",      "content": f"{SYSTEM_PROMPT}\n\n{row['prompt']}"},
            {"role": "assistant", "content": row["response"]},
        ]
    else:
        msgs = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": row["prompt"]},
            {"role": "assistant", "content": row["response"]},
        ]
    # TRL v1.4.0 does not re-add <bos> tokens when tokenizing the text field directly.
    # We leave the <bos> token generated by apply_chat_template intact.
    text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

# num_proc parallelises chat-template rendering — ~5-10x faster on Colab's 8 vCPUs.
ds = Dataset.from_list(raw_rows).map(
    to_chat,
    remove_columns=["prompt", "response"],
    num_proc=os.cpu_count() or 4,
)
print(f"Dataset built: {len(ds)} examples  |  fields: {ds.column_names}")
print("\nSample formatted text (truncated):")
print(ds[0]["text"][:500])
print("...")


Map (num_proc=12):   0%|          | 0/8367 [00:00<?, ? examples/s]

Dataset built: 8367 examples  |  fields: ['t', 'slice', 'actions_float', 'reward', 'supervise', 'text']

Sample formatted text (truncated):
<bos><|turn>user
You manage batteries in 3 buildings that share one grid meter. Each step, pick one action per building.

[Actions]
CHARGE_100, CHARGE_80, CHARGE_60, CHARGE_40, CHARGE_20, IDLE, DISCHARGE_20, DISCHARGE_40, DISCHARGE_60, DISCHARGE_80, DISCHARGE_100

[State]
- 'price' (LOW / PEAK): how expensive grid electricity is now.
- 'carbon' (LOW / MID / HIGH): how dirty grid electricity is now.
- 'solar' (NONE / LOW / MID / HIGH): the building's solar generation now.
- 'load' (kWh): the buil
...


In [ ]:
# Quick sanity: TRL is importable. Completion-only loss AND the per-cell
# no-op mask are applied by a custom tokenisation map in § 6 (see that cell)
# — not by Unsloth's train_on_responses_only, which cannot do the no-op mask.
import trl
print(f"trl=={trl.__version__}")


trl==1.4.0


In [ ]:
from google.colab import drive

# This will prompt you to authorize Google Drive access
drive.mount('/content/drive')

Mounted at /content/drive


## § 6 — Train (TRL `SFTTrainer` on Unsloth model)

Unsloth's patched model + tokenizer plug straight into TRL's `SFTTrainer`.

The loss labels are built by a custom tokenisation `.map` (`tokenize_and_label`
below) that applies **two masks**:

1. **Completion-only** — every token before the `<|turn>model` marker (the
   ~480-token system prompt) gets label `-100`, so cross-entropy is computed
   only on the assistant turn.
2. **No-op cells** — within the assistant turn, the `<action ...>` line of any
   building whose SAC action is a physical no-op (the `supervise` flag from
   § 3) also gets `-100`. Those cells carry an artifact `IDLE` label; the first
   run supervised them and collapsed to constant IDLE.

This replaces Unsloth's `train_on_responses_only` (mask 1 only). Token spans
are located via the tokenizer's char `offset_mapping`, so it is robust to the
Gemma-4 multimodal processor. The dataset fed to `SFTTrainer` is already
tokenised (`input_ids` / `labels`), so no `dataset_text_field` is set.


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
import re, time

# ── Build loss labels: completion-only mask + no-op-cell mask ────────────
# We construct input_ids / labels ourselves instead of using Unsloth's
# train_on_responses_only, because the loss needs a SECOND mask on top of the
# prompt mask: the <action> line of any building whose SAC move is a physical
# no-op (supervise=False, from § 3) must not contribute gradient. Supervising
# those artifact IDLE labels is what collapsed the previous run to constant
# IDLE. Spans are located via the tokenizer's char offset_mapping (robust to
# the Gemma-4 multimodal processor). text_tok is the fast text tokenizer
# set in § 5.
_ACTION_LINE_RE = re.compile(r"<action\s+building\s*=\s*(\d+)\s*>.*?</action>", re.S)
_RESP_MARKER    = "<|turn>model\n"          # Gemma-4 assistant-turn marker (§ 5)

def tokenize_and_label(row):
    text = row["text"]
    sup  = row["supervise"]                 # per-building bool, from § 3
    mi   = text.find(_RESP_MARKER)
    resp_start = mi + len(_RESP_MARKER) if mi != -1 else 0
    # char spans of <action> lines whose building is a no-op (mask these)
    masked = [(m.start(), m.end())
              for m in _ACTION_LINE_RE.finditer(text, resp_start)
              if int(m.group(1)) < len(sup) and not sup[int(m.group(1))]]
    enc = text_tok(text, add_special_tokens=False, return_offsets_mapping=True,
                   truncation=True, max_length=MAX_SEQ_LEN)
    ids    = enc["input_ids"]
    labels = []
    for tid, (cs, ce) in zip(ids, enc["offset_mapping"]):
        in_response = cs >= resp_start
        in_noop     = any(cs < me and ce > ms for ms, me in masked)
        labels.append(tid if (in_response and not in_noop) else -100)
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_tok = ds.map(tokenize_and_label, remove_columns=ds.column_names,
                   num_proc=os.cpu_count() or 4)
print(f"Tokenised {len(train_tok)} examples (input_ids / attention_mask / labels)")

# Save directly to Drive so a Colab disconnect doesn't nuke the run.
# Drive must be mounted (§ 5) BEFORE this cell.
OUT_DIR = f"/content/drive/MyDrive/eclipse-thesis/sft_out_v13/{time.strftime('%Y%m%d_%H%M%S')}"

sft_cfg = SFTConfig(
    output_dir                  = OUT_DIR,
    num_train_epochs            = EPOCHS,       # from § 0 config — loss flattens early, 1 is enough
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    max_length                  = MAX_SEQ_LEN,
    logging_steps               = 25,

    # ── Checkpointing → Drive (survives disconnects) ─────────────────────
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 3,            # rotate, don't fill Drive

    bf16                        = True,
    fp16                        = False,
    optim                       = "paged_adamw_8bit",
    warmup_steps                = 50,
    lr_scheduler_type           = "linear",
    report_to                   = "none",
    seed                        = SEED,
    # No dataset_text_field: train_tok is ALREADY tokenised (input_ids /
    # labels), so SFTTrainer skips its own tokenisation/masking.
    # Required on L4 (22GB) — Gemma-4 E4B + 4-bit + LoRA + bs 2 OOMs without it.
    gradient_checkpointing         = True,
    gradient_checkpointing_kwargs  = {"use_reentrant": False},
)

trainer = SFTTrainer(
    model           = model,
    args            = sft_cfg,
    train_dataset   = train_tok,
    processing_class= tokenizer,
    # Explicit collator: pads input_ids / attention_mask, pads labels with
    # -100. Removes any ambiguity in how a pre-tokenised dataset is batched.
    data_collator   = DataCollatorForSeq2Seq(text_tok, padding=True,
                                             label_pad_token_id=-100),
)

# ── Verify BOTH masks + single <bos> BEFORE the long train ───────────────
# Pick an example that actually HAS a masked (no-op) building, so the no-op
# mask is exercised — not just the completion-only mask.
_idx = next((i for i in range(len(ds)) if not all(ds[i]["supervise"])), 0)
_ex  = train_tok[_idx]
_sv  = ds[_idx]["supervise"]
_n_bos = sum(t == text_tok.bos_token_id for t in _ex["input_ids"])
_sup   = [t for t, l in zip(_ex["input_ids"], _ex["labels"]) if l != -100]
_span  = text_tok.decode(_sup)
print(f"sanity | example {_idx}  supervise={_sv}")
print(f"sanity | <bos> count       = {_n_bos}  (expect 1)")
print(f"sanity | supervised tokens = {len(_sup)}/{len(_ex['labels'])}")
print(f"sanity | supervised span   = {_span!r}")
assert _n_bos == 1, f"expected exactly one <bos>, got {_n_bos}"
assert 0 < len(_sup) < len(_ex["labels"]), \
    "completion-only mask failed (supervised span empty or full)"
for _b, _s in enumerate(_sv):
    in_loss = f"building={_b}" in _span
    assert in_loss == _s, (
        f"building {_b}: supervise={_s} but in-loss={in_loss} — no-op mask wrong"
    )
print("sanity | no-op mask OK — only supervised buildings contribute to the loss")

_t0 = time.time()
trainer.train()
print(f"\nSFT done in {time.time() - _t0:.1f}s")

# Save the final (last) model state — no eval split, so this is the
# last-step adapter. OUT_DIR is on Drive, so it survives runtime shutdown.
ADAPTER_DIR = f"{OUT_DIR}/lora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved: {ADAPTER_DIR}")


Map (num_proc=12):   0%|          | 0/8367 [00:00<?, ? examples/s]

Tokenised 8367 examples (input_ids / attention_mask / labels)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


sanity | example 0  supervise=[True, False, True]
sanity | <bos> count       = 1  (expect 1)
sanity | supervised tokens = 30/703
sanity | supervised span   = '<action building=0>CHARGE_20</action>\n\n<action building=2>CHARGE_20</action><turn|>\n'
sanity | no-op mask OK — only supervised buildings contribute to the loss


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,367 | Num Epochs = 2 | Total steps = 1,046
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 36,700,160 of 8,032,856,608 (0.46% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


...

Step,Training Loss
25,1.093772
50,0.030741
75,0.014184
100,0.012214
125,0.011610
150,0.010980
175,0.010973
200,0.011039
225,0.010160
250,0.010549


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/eclipse-thesis/sft_out_v13/20260518_124126/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/eclipse-thesis/sft_out_v13/20260518_124126/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/eclipse-thesis/sft_out_v13/20260518_124126/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/eclipse-thesis/sft_out_v13/20260518_124126/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/eclipse-thesis/sft_out_v13/20260518_124126/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/eclipse-thesis/sft_out_v13/20260518_124126/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDr


SFT done in 11427.0s


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/eclipse-thesis/sft_out_v13/20260518_124126/lora_adapter/tokenizer_config.json.


Adapter saved: /content/drive/MyDrive/eclipse-thesis/sft_out_v13/20260518_124126/lora_adapter


In [ ]:
import os
from unsloth import FastModel
import torch

# Point this to the directory where your adapter is saved on Drive
ADAPTER_DIR = "/content/drive/MyDrive/eclipse-thesis/sft_adaptersV13/lora_adapter" # Adjust if necessary

# Clear VRAM before loading
torch.cuda.empty_cache()

print(f"Loading model and adapter from: {ADAPTER_DIR}")
if os.path.exists(ADAPTER_DIR):
    # Load the model with the adapter applied
    model, tokenizer = FastModel.from_pretrained(
        model_name = ADAPTER_DIR,
        max_seq_length = MAX_SEQ_LEN,
        dtype = None,
        load_in_4bit = LOAD_IN_4BIT,
    )
    FastModel.for_inference(model) # enable 2x faster inference
    print("Model and adapter successfully loaded for inference!")
else:
    print(f"Error: Adapter directory not found at {ADAPTER_DIR}. Please check the path.")

Loading model and adapter from: /content/drive/MyDrive/eclipse-thesis/sft_adaptersV13/lora_adapter
==((====))==  Unsloth 2026.5.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Current model requires 6192 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Model and adapter successfully loaded for inference!


## § 6.5 — Post-Train Action-Token Diagnostics

The aggregate training loss (~0.007) is **boilerplate-dominated**: it averages
cross-entropy over the whole supervised span `<action building=N>TOKEN</action>`,
where most tokens are the deterministic XML scaffold the model fits to ~0 loss
instantly. It barely reflects decision quality.

This cell isolates the **action-magnitude token** (`CHARGE_20` / `IDLE` / ...)
and reports, on the training set (no held-out split — nb 06's rollout on
unseen buildings is the generalisation test):

- **per-action-token loss** — cross-entropy at the magnitude token only.
  Compare to `ln(11)=2.40` (uniform guess) and to the teacher's **marginal
  entropy** (the "always predict the base rate" floor). A loss below the
  marginal entropy means the model learned a real state->action map, not
  just the class frequencies.
- **action-token top-1 accuracy** — teacher-forced argmax vs the true
  teacher token; plus the coarse charge/idle/discharge accuracy.
- **predicted-token distribution** — doubles as a collapse check; one token
  predicted >85% of the time is the v6 failure mode. The cell asserts this,
  restoring the collapse gate that the rebuild dropped.

In [ ]:
# 6.5 - Post-train action-token diagnostics (training set; no held-out split).
import torch, re, math, collections

text_tok     = getattr(tokenizer, "tokenizer", tokenizer)
_RESP_MARKER = "<|turn>model\n"
_MAG_RE      = re.compile(r"<action\s+building\s*=\s*(\d+)\s*>(.*?)</action>", re.S)
_dev         = next(model.parameters()).device
# Stride across the whole year so the sample is not just early-January rows.
N_DIAG       = 300
_idxs        = list(range(0, len(ds), max(1, len(ds) // N_DIAG)))[:N_DIAG]

def _coarse(tok):
    if tok.startswith("CHARGE"):    return "CHARGE"
    if tok.startswith("DISCHARGE"): return "DISCHARGE"
    return "IDLE"

model.eval()
tok_losses, n_cells, n_exact, n_coarse = [], 0, 0, 0
true_ctr, pred_ctr = collections.Counter(), collections.Counter()

for idx in _idxs:
    row  = ds[idx]
    text = row["text"]
    sup  = row["supervise"]
    mi   = text.find(_RESP_MARKER)
    rs   = mi + len(_RESP_MARKER) if mi != -1 else 0
    # (char_start, char_end, true_token) of each magnitude string, supervised only
    mags = [(m.start(2), m.end(2), m.group(2).strip())
            for m in _MAG_RE.finditer(text, rs)
            if int(m.group(1)) < len(sup) and sup[int(m.group(1))]]
    if not mags:
        continue
    enc       = text_tok(text, add_special_tokens=False, return_offsets_mapping=True,
                         truncation=True, max_length=MAX_SEQ_LEN)
    ids, offs = enc["input_ids"], enc["offset_mapping"]
    with torch.no_grad():
        logits = model(torch.tensor([ids], device=_dev)).logits[0]
    for ms, me, true_tok in mags:
        # token indices overlapping the magnitude char span (logits[i-1] predicts i)
        span = [i for i, (cs, ce) in enumerate(offs)
                if i > 0 and cs < me and ce > ms and ce > cs]
        if not span:
            continue
        true_ctr[true_tok] += 1
        preds = []
        for i in span:
            lg = logits[i - 1].float()
            tok_losses.append(torch.nn.functional.cross_entropy(
                lg.unsqueeze(0), torch.tensor([ids[i]], device=_dev)).item())
            preds.append(int(lg.argmax()))
        pred_tok = text_tok.decode(preds).strip()
        pred_ctr[pred_tok] += 1
        n_cells  += 1
        n_exact  += int(pred_tok == true_tok)
        n_coarse += int(_coarse(pred_tok) == _coarse(true_tok))
    del logits

# Marginal entropy of the teacher's action mix - the "predict the base rate" floor.
_tot   = sum(true_ctr.values())
marg_H = -sum((c / _tot) * math.log(c / _tot) for c in true_ctr.values())
maj_t, maj_n = true_ctr.most_common(1)[0]

print(f"Diagnosed {n_cells} supervised action cells from {len(_idxs)} rows\n")
print(f"per-action-token loss (mean CE) : {sum(tok_losses) / len(tok_losses):.4f}")
print(f"  ref - uniform guess (ln 11)   : {math.log(11):.4f}")
print(f"  ref - marginal entropy        : {marg_H:.4f}  (beat this = real mapping)")
print(f"action-token top-1 accuracy     : {n_exact / n_cells:6.1%}")
print(f"  majority-class baseline       : {maj_n / _tot:6.1%}  (always '{maj_t}')")
print(f"coarse charge/idle/discharge acc: {n_coarse / n_cells:6.1%}\n")
print("predicted-token distribution (collapse check):")
for tok, c in pred_ctr.most_common():
    print(f"  {tok:16s} {c / n_cells:6.1%}")
_top = pred_ctr.most_common(1)[0]
_ok  = _top[1] / n_cells < 0.85
print(f"\n{'OK' if _ok else 'COLLAPSE'} - top predicted token '{_top[0]}' = "
      f"{_top[1] / n_cells:.1%}  (gate: < 85%)")
assert _ok, "degenerate run - model collapsed to one action token"

## § 12 — Persist Adapter to Drive (optional)

In [ ]:
# Ensure the target directory exists
!mkdir -p /content/drive/MyDrive/eclipse-thesis/sft_adaptersV13/

# Copy the saved adapter to your Drive
!cp -r {ADAPTER_DIR} /content/drive/MyDrive/eclipse-thesis/sft_adaptersV13/

print(f"\nAdapter successfully backed up to Google Drive!")
print(f"Look for it in: MyDrive/eclipse-thesis/sft_adapters/")


Adapter successfully backed up to Google Drive!
Look for it in: MyDrive/eclipse-thesis/sft_adapters/


## § 7 — Build Inference Provider (Unsloth fast inference)

`FastModel.for_inference(model)` enables Unsloth's 2× faster inference path
(disables training-only autograd hooks). Same `.complete()` / `.step()` API
as notebook 03's `LocalHFProvider`, so the rollout helper below is unchanged.

In [ ]:
import logging, re, inspect, textwrap
import transformers.processing_utils
from unsloth.chat_templates import get_chat_template
from src.sft import parse_actions, make_sft_prompt

# ── Inference fix (mirrors nb 06's working setup) ─────────────────────────
# The § 12 reload returns a FRESH tokenizer without the gemma-4 chat template,
# on the un-patched transformers processor. Both must be restored here or
# SFTProvider.complete() renders the wrong template / crashes mid-rollout
# (the symptom: § 10 stalls right after printing the SFT-SLM header).
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

# Gemma-4's multimodal ProcessorMixin.apply_chat_template crashes on plain
# string message content (it evaluates content["type"] on a str). Patch the
# type guard — same fix nb 06 applies in its install section.
_src = textwrap.dedent(inspect.getsource(
    transformers.processing_utils.ProcessorMixin.apply_chat_template))
_src = _src.replace(
    'if content["type"] in ["image", "video"]',
    'if isinstance(content, dict) and content.get("type") in ["image", "video"]')
_g = transformers.processing_utils.__dict__.copy()
exec(_src, _g)
transformers.processing_utils.ProcessorMixin.apply_chat_template = _g["apply_chat_template"]
print("inference fix: gemma-4 chat template re-applied + apply_chat_template patched")

# Define N_BUILDINGS since we skipped the dataset generation cell
N_BUILDINGS = 3

_logger = logging.getLogger("sft_slm")
# Opening-tag-only regex — DIFFERENT from src.agent.ACTION_RE (which captures the
# full <action ...>TOKEN</action> for parsing). This one is used elsewhere for
# a quick "did the model emit any action tags?" check.
_ACTION_TAG_RE = re.compile(r"<action\s+building\s*=\s*\d+\s*>", re.IGNORECASE)

FastModel.for_inference(model)  # 2× faster inference


class SFTProvider:
    """Inference provider wrapping the LoRA-fine-tuned Unsloth model."""

    def __init__(self, model, tokenizer, max_new_tokens: int = 200):
        self.model         = model.eval()
        self.tokenizer     = tokenizer
        self.max_new_tokens= max_new_tokens
        self.label         = f"sft:{MODEL_ID.split('/')[-1]}"
        self._is_gemma     = "gemma" in MODEL_ID.lower()
        self._device       = next(model.parameters()).device

    def complete(self, system: str, user: str, max_tokens=None) -> str:
        max_new = max_tokens or self.max_new_tokens
        if self._is_gemma:
            msgs = [{"role": "user", "content": f"{system}\n\n{user}"}]
        else:
            msgs = [{"role": "system", "content": system},
                    {"role": "user",   "content": user}]

        # Bypass transformers multimodal chat template bug by rendering to string first
        prompt_str = self.tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )

        # Explicitly pass text=prompt_str because multimodal processors
        # treat the first positional argument as 'images'
        enc = self.tokenizer(text=prompt_str, return_tensors="pt").to(self._device)

        with torch.no_grad():
            out = self.model.generate(
                **enc,
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
            )
        new_tokens = out[0][enc["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def step(self, state_text: str, system=None, n_buildings: int = 6,
             max_retries: int = 2, **kw):
        sys_p = system or make_sft_prompt(n_buildings)
        last  = ""
        for _ in range(max_retries):
            last = self.complete(sys_p, f"STATE:\n{state_text}")
            if _ACTION_TAG_RE.search(last):
                return parse_actions(last, n_buildings), last, False
        return [0.0]*n_buildings, last, True


slm = SFTProvider(model, tokenizer, max_new_tokens=MAX_NEW_TOKENS)
print(f"Provider ready: {slm.label}")

Provider ready: sft:gemma-4-E4B-it


## § 8 — Evaluation Environment Factory (Colab)

In [ ]:
import citylearn
from citylearn.citylearn import CityLearnEnv
from citylearn.agents.rbc import BasicBatteryRBC
from src.env import (
    make_env, snapshot_state, MERLINReward,
    OBSERVATIONS, ACTIVE_ACTIONS,
    TRAINING_BUILDINGS, HELDOUT_BUILDINGS, BUILDINGS,
)
from src.eval import evaluate, comparison_table
from src.sft  import render_state

warnings.filterwarnings("ignore")
np.random.seed(SEED); random.seed(SEED)

# Default to the 3-building TRAINING slice — this matches N_BUILDINGS=3 in
# cell 15 and the slice geometry the JSONL was emitted on. Previously this
# defaulted to BUILDINGS=[0..5], which made the headline eval below a 6-bldg
# OOD rollout that the SLM had never been trained for.
#
# Schema source: use the project-wide src.env.make_env so this notebook hits
# the SAME local-path schema as nb 04 (dataset dump) — the repo was cloned
# in § 2 so the data/ tree is on disk. Previously this notebook constructed
# CityLearnEnv directly with `schema="citylearn_challenge_2022_phase_all"`,
# which triggers a separate download from the pinned CityLearn tag and may
# drift from the file nb 04 used.
def make_colab_env(start=0, end=8759, buildings=None) -> CityLearnEnv:
    return make_env(
        buildings = buildings if buildings is not None else TRAINING_BUILDINGS,
        start     = start,
        end       = end
    )

print(f"CityLearn {citylearn.__version__}  |  default eval buildings = TRAINING_BUILDINGS = {TRAINING_BUILDINGS}")

CityLearn 2.6.0b2  |  default eval buildings = TRAINING_BUILDINGS = [0, 1, 2]


## § 9 — Rollout Helpers

In [ ]:
def run_slm_policy(name: str, provider, start=EVAL_START, length=EVAL_LEN,
                   n_buildings: int = N_BUILDINGS, verbose_every: int = 24):
    env = make_colab_env(start=start, end=start+length-1)
    env.reset()
    sys_p = make_sft_prompt(n_buildings)
    done, t, t0, n_fb = False, 0, time.time(), 0
    while not done:
        snap = snapshot_state(env)
        state_text = render_state(snap)
        acts, raw, fb = provider.step(state_text, system=sys_p, n_buildings=n_buildings)
        n_fb += int(fb)
        _, _, term, trunc, _ = env.step([[float(a)] for a in acts])
        done = bool(term or trunc)
        if verbose_every and (t % verbose_every == 0):
            elapsed = time.time() - t0
            print(f"  t={t:4d} | {elapsed:.0f}s | fallbacks={n_fb}")
        t += 1
    print(f"[{name}] {t} steps in {time.time()-t0:.1f}s | fallbacks={n_fb}/{t}")
    return env


def run_rbc(start=EVAL_START, length=EVAL_LEN):
    env = make_colab_env(start=start, end=start+length-1)
    BasicBatteryRBC(env=env).learn(episodes=1)
    return env


def run_noop(start=EVAL_START, length=EVAL_LEN):
    env = make_colab_env(start=start, end=start+length-1)
    env.reset()
    n_b = len(env.buildings)
    done = False
    while not done:
        _, _, term, trunc, _ = env.step([[0.0] for _ in range(n_b)])
        done = bool(term or trunc)
    return env

## § 10 — Run Baselines + SFT-SLM

In [ ]:
print("── No-control baseline ──")
env_noop = run_noop()

print("\n── RBC ──")
env_rbc  = run_rbc()

print(f"\n── SFT SLM ({slm.label}) ──")
env_slm  = run_slm_policy("sft_slm", slm)

── No-control baseline ──

── RBC ──

── SFT SLM (sft:gemma-4-E4B-it) ──


## § 11 — Evaluation

In [ ]:
# Challenge KPIs via src.eval (evaluate_v2 — CityLearn 2.6+)
eval_results = [
    evaluate(env_noop, label="No-Control"),
    evaluate(env_rbc,  label="RBC"),
    evaluate(env_slm,  label=slm.label),
]
challenge_df, zne_df = comparison_table(eval_results)
print("Challenge scores — lower is better; No-Control is the reference baseline.")
display(challenge_df.round(4))

print("\nZNE + self-consumption:")
display(zne_df[["solar generation (kWh)", "grid import (kWh)",
                "ZNE ratio (solar / import)", "self-consumption ratio"]].round(4))


## § 13 — Battery State of Charge (SOC) Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_buildings = len(env_slm.buildings)
fig, axes = plt.subplots(nrows=n_buildings, ncols=1, figsize=(12, 2 * n_buildings), sharex=True)

if n_buildings == 1:
    axes = [axes]

for i, b in enumerate(env_slm.buildings):
    # Extract the SOC history for the episode
    soc_history = b.electrical_storage.soc

    axes[i].plot(soc_history, label=f'Building {i}', color='tab:blue', linewidth=1.5)
    axes[i].set_ylabel('SOC')
    axes[i].set_ylim(-0.05, 1.05)  # SOC is strictly between 0 and 1
    axes[i].legend(loc='upper right')
    axes[i].grid(True, linestyle='--', alpha=0.7)

axes[-1].set_xlabel('Time Step (Hours)')
fig.suptitle(f'Battery State of Charge (SOC) over 1 Week - {slm.label}', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

## § 14 — Debugging: Inspect Raw SLM Output
Let's run a single step to see exactly what the model is generating.

In [ ]:
from src.env import snapshot_state
from src.sft import render_state, make_sft_prompt

# Create a test environment and get the first state
env_test = make_colab_env(start=EVAL_START, end=EVAL_START+5)
env_test.reset()
snap = snapshot_state(env_test)
state_text = render_state(snap)
sys_p = make_sft_prompt(N_BUILDINGS)

print("=== RAW SLM GENERATION TEST ===\n")
print("--- PROMPT STATE ---")
print(state_text[:500] + "\n... [truncated]")

print("\n--- MODEL RESPONSE ---")
# Run the complete method to get the raw string
raw_resp = slm.complete(sys_p, f"STATE:\n{state_text}")
print(raw_resp)

print("\n--- PARSED ACTIONS ---")
acts, raw, fallback = slm.step(state_text, system=sys_p, n_buildings=N_BUILDINGS)
print(f"Parsed actions: {acts}")
print(f"Did it trigger a fallback (parsing error)? {fallback}")

## § 15 — 1-Day Inference Trace
Run a 24-hour rollout and print the exact states and responses for inspection.

In [ ]:
print("=== 1-DAY INFERENCE TRACE ===")
# 24 steps for a single day
env_day = make_colab_env(start=EVAL_START, end=EVAL_START+23)
env_day.reset()
sys_p = make_sft_prompt(N_BUILDINGS)

done = False
step_idx = 0

while not done:
    print(f"\n{'='*40}")
    print(f" STEP {step_idx:02d}")
    print(f"{'='*40}")

    snap = snapshot_state(env_day)
    state_text = render_state(snap)

    print("[INPUT STATE]")
    # Printing the first few lines to keep output readable,
    # but we can print the whole thing if preferred.
    lines = state_text.split('\n')
    print('\n'.join(lines[:4]))
    print("  ... [buildings truncated] ...")

    acts, raw_resp, fallback = slm.step(state_text, system=sys_p, n_buildings=N_BUILDINGS)

    print("\n[MODEL OUTPUT]")
    print(raw_resp.strip())
    print(f"\n-> Parsed Actions: {acts} (Fallback: {fallback})")

    _, _, term, trunc, _ = env_day.step([[float(a)] for a in acts])
    done = bool(term or trunc)
    step_idx += 1


## § 16 — Inspect the Exact Inference Prompt
Print the fully formatted prompt (with chat template tokens) sent to the model.

In [ ]:
from src.env import snapshot_state
from src.sft import render_state, make_sft_prompt

# Get step 0 state
env_test = make_colab_env(start=EVAL_START, end=EVAL_START+1)
env_test.reset()
snap = snapshot_state(env_test)
state_text = render_state(snap)

sys_p = make_sft_prompt(N_BUILDINGS)
user_p = f"STATE:\n{state_text}"

# Combine them exactly as SFTProvider.complete() does for Gemma
msgs = [{"role": "user", "content": f"{sys_p}\n\n{user_p}"}]

# Apply the chat template to see the raw input string
raw_prompt = slm.tokenizer.apply_chat_template(
    msgs,
    tokenize=False,
    add_generation_prompt=True
)

print("=== EXACT MODEL INPUT PROMPT ===\n")
print(raw_prompt)

## § 17 — Inference with CoT Minimal Prompt
Testing the modified prompt that introduces Chain-of-Thought `<thought>` blocks and specific operational constraints.

In [ ]:
# CoT prompt for the optional CoT-eval section comes from src.agent
# (canonical CoT prompt). The fine-tuned SLM was trained on make_sft_prompt
# (no <thought> block), so applying make_minimal_prompt at eval time is OOD —
# this section exists to QUANTIFY that drift, not to be used as the headline KPI.
from src.agent import make_minimal_prompt

sys_p_cot = make_minimal_prompt(N_BUILDINGS)

print('CoT prompt for OOD comparison:')
print(sys_p_cot)


## § 18 — Verify CoT Prompt Format
Let's print the exact chat-formatted string that gets sent to the model during the CoT inference to confirm the state is included.

In [ ]:
# Combine the CoT system prompt and the current state
msgs = [{"role": "user", "content": f"{sys_p_cot}\n\nSTATE:\n{state_text}"}]

# Apply the Gemma chat template
raw_cot_prompt = slm.tokenizer.apply_chat_template(
    msgs,
    tokenize=False,
    add_generation_prompt=True
)

print("=== EXACT CoT MODEL INPUT PROMPT ===\n")
print(raw_cot_prompt)

## § 19 — 1-Day Evaluation with CoT Prompt
Let's run a 24-hour episode using the CoT prompt to see if the reasoning step improves battery utilization.

In [ ]:
import time
from src.env import snapshot_state
from src.sft import render_state
import pandas as pd
import matplotlib.pyplot as plt

print("=== 1-DAY CoT EVALUATION ===")
# 24 steps = 1 day
env_cot = make_colab_env(start=EVAL_START, end=EVAL_START+23)
env_cot.reset()

done, t, t0, n_fb = False, 0, time.time(), 0

while not done:
    snap = snapshot_state(env_cot)
    state_text = render_state(snap)

    # Use the CoT system prompt and allow more tokens for the thought block
    acts, raw, fb = slm.step(state_text, system=sys_p_cot, n_buildings=N_BUILDINGS, max_tokens=250)

    n_fb += int(fb)
    _, _, term, trunc, _ = env_cot.step([[float(a)] for a in acts])
    done = bool(term or trunc)

    if t % 6 == 0:
        print(f"  t={t:2d} | fallbacks={n_fb}")
    t += 1

print(f"Done in {time.time()-t0:.1f}s | fallbacks={n_fb}/{t}")

# Evaluate KPIs via src.eval (evaluate_v2)
res_cot = evaluate(env_cot, label='CoT (1 Day)')
display(res_cot.challenge.round(4))
display(res_cot.zne[['solar generation (kWh)', 'grid import (kWh)',
                     'ZNE ratio (solar / import)', 'self-consumption ratio']].round(4))

# Plot SOC
fig, axes = plt.subplots(nrows=N_BUILDINGS, ncols=1, figsize=(12, 2 * N_BUILDINGS), sharex=True)
if N_BUILDINGS == 1: axes = [axes]

for i, b in enumerate(env_cot.buildings):
    axes[i].plot(b.electrical_storage.soc, label=f'Building {i}', color='tab:orange', linewidth=2)
    axes[i].set_ylabel('SOC')
    axes[i].set_ylim(-0.05, 1.05)
    axes[i].legend(loc='upper right')
    axes[i].grid(True, linestyle='--', alpha=0.7)

axes[-1].set_xlabel('Time Step (Hours)')
fig.suptitle('Battery SOC over 1 Day - CoT Prompt', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()


## § 20 — 1-Day Inference Trace (CoT Prompt)
Run a 24-hour rollout using the Chain-of-Thought prompt and print the exact states and responses for inspection.

In [ ]:
print("=== 1-DAY CoT INFERENCE TRACE ===")
# 24 steps for a single day
env_cot_trace = make_colab_env(start=EVAL_START, end=EVAL_START+23)
env_cot_trace.reset()

done = False
step_idx = 0

while not done:
    print(f"\n{'='*40}")
    print(f" STEP {step_idx:02d}")
    print(f"{'='*40}")

    snap = snapshot_state(env_cot_trace)
    state_text = render_state(snap)

    print("[INPUT STATE]")
    lines = state_text.split('\n')
    print('\n'.join(lines[:4]))
    print("  ... [buildings truncated] ...")

    # Use sys_p_cot and allow more max_tokens for the reasoning block
    acts, raw_resp, fallback = slm.step(state_text, system=sys_p_cot, n_buildings=N_BUILDINGS, max_tokens=250)

    print("\n[MODEL OUTPUT (CoT)]")
    print(raw_resp.strip())
    print(f"\n-> Parsed Actions: {acts} (Fallback: {fallback})")

    _, _, term, trunc, _ = env_cot_trace.step([[float(a)] for a in acts])
    done = bool(term or trunc)
    step_idx += 1
